# 1 — Dados e rótulos da GAN condicional

Objetivo: conhecer as imagens e a **condição** que acompanha cada exemplo. Execute as células de cima para baixo. O primeiro download precisa de internet. Nenhum modelo é treinado neste notebook.

In [ ]:
from pathlib import Path
import json, csv, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Funciona abrindo o Jupyter na raiz ou na pasta notebooks.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modelos.py").exists())
CLASSES = ['Camiseta/Top', 'Calça', 'Pulôver', 'Vestido', 'Casaco',
           'Sandália', 'Camisa', 'Tênis', 'Bolsa', 'Bota']
SEMENTE = 42
torch.set_num_threads(2)
torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Raiz:', RAIZ, '| Dispositivo:', DISPOSITIVO)


## Preparação e separação
50 mil imagens para treino, 10 mil para validação e 10 mil de teste oficial. Somente treino atualiza as redes. Normalizamos pixels para [-1, 1], como a saída `Tanh` do gerador.

In [ ]:
transformacao = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
base = datasets.FashionMNIST(str(RAIZ / 'data'), train=True, download=True, transform=transformacao)
teste = datasets.FashionMNIST(str(RAIZ / 'data'), train=False, download=True, transform=transformacao)
idx_treino, idx_validacao = train_test_split(
    np.arange(len(base)), train_size=50000, test_size=10000,
    stratify=base.targets.numpy(), random_state=SEMENTE)
treino, validacao = Subset(base, idx_treino), Subset(base, idx_validacao)
assert not set(idx_treino).intersection(idx_validacao)
assert len(treino) == 50000 and len(validacao) == len(teste) == 10000
print('Treino, validação, teste:', len(treino), len(validacao), len(teste))
print('Treino por classe:', np.bincount(base.targets[idx_treino]))


## Imagem e categoria são um par
A cGAN recebe o rótulo tanto no gerador quanto no discriminador. Não se trata de classificar uma imagem depois de gerá-la.

In [ ]:
fig, eixos = plt.subplots(2, 5, figsize=(12, 5))
for classe, eixo in enumerate(eixos.flat):
    indice = next(int(i) for i in idx_treino if int(base.targets[i]) == classe)
    imagem, rotulo = base[indice]
    eixo.imshow((imagem[0] + 1) / 2, cmap='gray', vmin=0, vmax=1)
    eixo.set_title(f'{rotulo}: {CLASSES[rotulo]}'); eixo.axis('off')
plt.tight_layout(); plt.show()


## Para a apostila
Explique a diferença entre um pixel e um rótulo. A categoria é uma informação adicional usada no aprendizado; não garante que toda imagem gerada já corresponda visualmente ao pedido. Validação e teste ficam reservados; não calculamos métricas de qualidade neste fluxo.